# 🛡️ GAN-Based Adversarial DDoS Traffic Generation for IDS Robustness Testing

**Mô tả:** Sử dụng 3 kiến trúc GAN (Vanilla GAN · WGAN-GP · LSGAN) để sinh mẫu DDoS đối nghịch, kiểm tra và đánh giá độ bền của IDS.

## Pipeline
```
Load Data → Clean → Imbalance Analysis → Feature Selection → Balancing
    ↓
Phase 1 : Baseline IDS (real data)              → Accuracy ~90%+
    ↓
Phase 2 : Train 3 GANs → Compare → Best GAN
    ↓
Phase 3 : Sinh mẫu adversarial với Best GAN
    ↓
Phase 4 : Test IDS trên adversarial data         → Accuracy ~70%
    ↓
Kết luận : So sánh Before vs After
```

| | |
|---|---|
| **Dataset** | CICIDS2017 |
| **GANs** | Vanilla GAN · WGAN-GP · LSGAN |
| **Classifier (IDS)** | Random Forest (`class_weight='balanced'`) |
| **Imbalance** | RandomUnderSampler + SMOTE |


---
## 1. Install & Import

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

required = [
    'numpy', 'pandas', 'matplotlib', 'seaborn',
    'scikit-learn', 'tensorflow', 'imbalanced-learn', 'tqdm', 'kagglehub'
]
for pkg in required:
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        print(f'Installing {pkg}...')
        pip_install(pkg)

print('✅ All packages ready!')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
from collections import Counter
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             classification_report, confusion_matrix,
                             precision_recall_curve, average_precision_score)
from sklearn.decomposition import PCA
from scipy.stats import ks_2samp
from scipy.special import rel_entr
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.optimizers import Adam

np.random.seed(42)
tf.random.set_seed(42)
print(f'✅ NumPy {np.__version__} | TensorFlow {tf.__version__}')

---
## 2. Load CICIDS2017 Dataset

In [ ]:
import kagglehub

print('📥 Downloading CICIDS2017...')
dataset_path = kagglehub.dataset_download('mdalamintalukder/cicids2017')
print(f'   Path: {dataset_path}')

csv_files = []
for root, _, files in os.walk(dataset_path):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))
print(f'   Found {len(csv_files)} CSV files')

dfs = []
for fp in tqdm(csv_files, desc='Loading'):
    try:
        tmp = pd.read_csv(fp, encoding='utf-8', low_memory=False)
        dfs.append(tmp)
        print(f'   ✅ {os.path.basename(fp)}: {tmp.shape[0]:,} rows')
    except Exception as e:
        print(f'   ❌ {os.path.basename(fp)}: {e}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\n📊 Combined shape: {df_raw.shape}')

---
## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Chuẩn hóa tên cột
df.columns = df.columns.str.strip().str.replace(' ','_').str.replace('/','_')
label_col = [c for c in df.columns if 'label' in c.lower()][0]
print(f'Label column: "{label_col}"')

# Xử lý Inf / NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
n_miss = df.isnull().sum().sum()
df.dropna(inplace=True)
print(f'Dropped {n_miss:,} missing/inf values')

# Xóa duplicate
n_dup = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f'Dropped {n_dup:,} duplicates')

# ── Hiển thị tất cả loại nhãn gốc trong dataset ──────────────────
print(f'\n🏷️  Tất cả loại traffic trong dataset:')
all_labels = df[label_col].str.strip().value_counts()
for lbl, cnt in all_labels.items():
    print(f'   {lbl:<35s} : {cnt:>10,}')

# ── Lọc Benign + TẤT CẢ loại DoS/DDoS ──────────────────────────
# FIX: dùng regex 'dos' thay 'ddos' để bắt cả DoS GoldenEye/Hulk/Slowloris/Slowhttptest/DDoS
df[label_col] = df[label_col].str.strip()
df['Label_Original'] = df[label_col].copy()

df['Label_Binary'] = df[label_col].apply(
    lambda x: 0 if 'benign' in str(x).lower() else 1
)
df = df[df[label_col].str.lower().str.contains(r'benign|dos')].copy()
df.reset_index(drop=True, inplace=True)

# ── Thống kê phân lớp sau khi lọc ────────────────────────────────
print(f'\n📊 Các loại DoS/DDoS được giữ lại:')
dos_types = df[df['Label_Binary']==1]['Label_Original'].value_counts()
for lbl, cnt in dos_types.items():
    print(f'   {lbl:<35s} : {cnt:>10,}')

cnt = df['Label_Binary'].value_counts()
total = len(df)
print(f'\n📊 Class distribution (Binary):')
print(f'   Benign : {cnt.get(0,0):>10,} ({cnt.get(0,0)/total*100:.1f}%)')
print(f'   DoS/DDoS: {cnt.get(1,0):>10,} ({cnt.get(1,0)/total*100:.1f}%)')
print(f'   Total  : {total:>10,}')
print(f'   Imbalance ratio: {cnt.get(0,0)/max(cnt.get(1,1),1):.2f}:1')


---
## 4. Class Imbalance Analysis

> Dữ liệu mất cân bằng cần được phân tích kỹ trước khi áp dụng kỹ thuật cân bằng.
> **Chiến lược:** `RandomUnderSampler` (giảm Benign) → `SMOTE` (tăng DDoS) → `class_weight='balanced'` trong classifier.

In [ ]:
cnt = df['Label_Binary'].value_counts()
imbalance_ratio = cnt[0] / cnt[1] if 1 in cnt else 0

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ── Plot 1: Bar chart binary ─────────────────────────────────────
colors = ['#4c72b0', '#c44e52']
bars = axes[0].bar(['Benign','DoS/DDoS'], [cnt.get(0,0), cnt.get(1,0)],
                   color=colors, edgecolor='k', linewidth=1.2)
for bar, val in zip(bars, [cnt.get(0,0), cnt.get(1,0)]):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02,
                f'{val:,}', ha='center', fontweight='bold', fontsize=11)
axes[0].set_title(f'Binary Class Distribution\n(Imbalance ratio {imbalance_ratio:.1f}:1)', fontsize=13)
axes[0].set_ylabel('Samples')

# ── Plot 2: Pie chart ────────────────────────────────────────────
axes[1].pie([cnt.get(0,0), cnt.get(1,0)], labels=['Benign','DoS/DDoS'],
            colors=colors, autopct='%1.1f%%', startangle=90,
            explode=[0, 0.06], textprops={'fontsize':12})
axes[1].set_title('Class Proportion', fontsize=13)

# ── Plot 3: Breakdown từng loại DoS/DDoS gốc ────────────────────
if 'Label_Original' in df.columns:
    dos_counts = df[df['Label_Binary']==1]['Label_Original'].value_counts()
    benign_cnt = pd.Series({'BENIGN': cnt.get(0,0)})
    all_counts = pd.concat([benign_cnt, dos_counts]).sort_values(ascending=True)
    
    bar_colors_multi = ['#4c72b0' if 'benign' in str(l).lower() else '#c44e52' 
                        for l in all_counts.index]
    bars3 = axes[2].barh(range(len(all_counts)), all_counts.values, 
                          color=bar_colors_multi, edgecolor='k', linewidth=0.8)
    axes[2].set_yticks(range(len(all_counts)))
    axes[2].set_yticklabels(all_counts.index, fontsize=10)
    axes[2].set_title('Phân loại loại tấn công gốc (CICIDS2017)', fontsize=12, fontweight='bold')
    axes[2].set_xlabel('Số lượng mẫu')
    for i, (bar, v) in enumerate(zip(bars3, all_counts.values)):
        axes[2].text(v + all_counts.max()*0.01, i, f'{v:,}', va='center', fontsize=9)

plt.suptitle('Class Imbalance Analysis – CICIDS2017 (All DoS/DDoS Types)', 
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Cảnh báo mức độ mất cân bằng ────────────────────────────────
if imbalance_ratio > 5:
    print(f'⚠️  Mất cân bằng nghiêm trọng ({imbalance_ratio:.1f}:1)')
    print('   → Sẽ áp dụng: RandomUnderSampler + SMOTE + class_weight="balanced"')
elif imbalance_ratio > 2:
    print(f'⚠️  Mất cân bằng vừa phải ({imbalance_ratio:.1f}:1)')
    print('   → Sẽ áp dụng: SMOTE + class_weight="balanced"')
else:
    print(f'✅ Mất cân bằng nhẹ ({imbalance_ratio:.1f}:1) – SMOTE only')


---
## 5. Feature Selection (Random Forest Importance)

In [ ]:
exclude = [label_col, 'Label_Binary']
feature_cols = [c for c in df.columns
                if c not in exclude
                and df[c].dtype in ['float64','int64','float32','int32']]
print(f'Numeric features available: {len(feature_cols)}')

# Sample để tăng tốc
sample_n = min(30_000, len(df))
df_s = df.sample(sample_n, random_state=42)
X_fs = df_s[feature_cols].values
y_fs = df_s['Label_Binary'].values

print('🌳 Computing RF feature importance...')
rf_fs = RandomForestClassifier(n_estimators=50, max_depth=8,
                               class_weight='balanced',
                               random_state=42, n_jobs=-1)
rf_fs.fit(X_fs, y_fs)
importances = pd.Series(rf_fs.feature_importances_, index=feature_cols)

N_TOP = 20
SELECTED_FEATURES = importances.nlargest(N_TOP).index.tolist()
print(f'\n✅ Top {N_TOP} selected features:')
for i, feat in enumerate(SELECTED_FEATURES, 1):
    print(f'   {i:2d}. {feat:42s} {importances[feat]:.4f}')

fig, ax = plt.subplots(figsize=(11, 6))
importances[SELECTED_FEATURES].sort_values().plot(
    kind='barh', ax=ax, color='steelblue', edgecolor='k')
ax.set_title('Top 20 Features – RF Importance', fontsize=14)
ax.set_xlabel('Importance Score')
plt.tight_layout(); plt.show()

---
## 6. Class Balancing – 2-Step Strategy

**Bước 1:** `RandomUnderSampler` — giảm Benign xuống tỉ lệ 2:1 (tránh mất quá nhiều thông tin DDoS)

**Bước 2:** `SMOTE` — tăng DDoS lên cân bằng 1:1 bằng cách sinh mẫu tổng hợp

> 💡 Kết hợp cả hai tốt hơn chỉ dùng SMOTE đơn lẻ với dữ liệu mất cân bằng nặng.

In [ ]:
X_raw = df[SELECTED_FEATURES].values
y_raw = df['Label_Binary'].values

cnt_before = Counter(y_raw)
print(f'Before balancing : {dict(cnt_before)}')

# Bước 1: UnderSampler – Benign giảm xuống còn 2× DDoS
n_ddos = cnt_before[1]
under_ratio = {0: min(cnt_before[0], n_ddos * 2), 1: n_ddos}
rus = RandomUnderSampler(sampling_strategy=under_ratio, random_state=42)
X_under, y_under = rus.fit_resample(X_raw, y_raw)
print(f'After Undersample : {dict(Counter(y_under))}')

# Bước 2: SMOTE – cân bằng hoàn toàn
smote = SMOTE(sampling_strategy=1.0, random_state=42, k_neighbors=5)
X_balanced, y_balanced = smote.fit_resample(X_under, y_under)
cnt_after = Counter(y_balanced)
print(f'After SMOTE       : {dict(cnt_after)}')

# Visualize
stages = ['Original', 'After UnderSampling', 'After SMOTE']
benign_counts = [cnt_before[0], Counter(y_under)[0], cnt_after[0]]
ddos_counts   = [cnt_before[1], Counter(y_under)[1], cnt_after[1]]

x = np.arange(len(stages)); w = 0.35
fig, ax = plt.subplots(figsize=(11, 5))
b1 = ax.bar(x - w/2, benign_counts, w, label='Benign', color='#4c72b0', edgecolor='k')
b2 = ax.bar(x + w/2, ddos_counts,   w, label='DDoS',   color='#c44e52', edgecolor='k')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
            f'{int(bar.get_height()):,}', ha='center', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(stages, fontsize=11)
ax.set_ylabel('Samples'); ax.legend(fontsize=11)
ax.set_title('Class Balancing Progress (2-Step Strategy)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print(f'\n✅ Final balanced dataset: {len(X_balanced):,} samples')

---
## Phase 1: Baseline IDS – Trained on Real Data

> **Mục tiêu:** Accuracy ≥ 90% trên tập test thật

> `class_weight='balanced'` giúp classifier không bị bias về lớp đa số.

In [ ]:
# Scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_balanced)

# Split
X_train, X_test_real, y_train, y_test_real = train_test_split(
    X_scaled, y_balanced,
    test_size=0.2, random_state=42, stratify=y_balanced
)

# Dữ liệu DDoS thật (unscaled) để train GAN
df_ddos_gan = df[df['Label_Binary'] == 1][SELECTED_FEATURES].copy()
print(f'Real DDoS for GAN training : {len(df_ddos_gan):,} samples')
print(f'Train set : {len(X_train):,} | Test set : {len(X_test_real):,}')

# ── Train ──
print('\n🤖 Training Baseline IDS (RF, class_weight=balanced)...')
ids_base = RandomForestClassifier(
    n_estimators=100, max_depth=15,
    class_weight='balanced',
    random_state=42, n_jobs=-1
)
ids_base.fit(X_train, y_train)

# ── Evaluate ──
y_pred_base = ids_base.predict(X_test_real)
y_prob_base = ids_base.predict_proba(X_test_real)[:, 1]

ACC_BASE = accuracy_score(y_test_real, y_pred_base)
F1_BASE  = f1_score(y_test_real, y_pred_base, average='macro')
AUC_BASE = roc_auc_score(y_test_real, y_prob_base)

print('=' * 60)
print('PHASE 1 RESULTS – Baseline IDS on Real Test Data')
print('=' * 60)
print(f'  Accuracy  : {ACC_BASE*100:.2f}%')
print(f'  F1 (macro): {F1_BASE:.4f}')
print(f'  AUC-ROC   : {AUC_BASE:.4f}')
print()
print(classification_report(y_test_real, y_pred_base, target_names=['Benign','DDoS']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test_real, y_pred_base)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Benign','DDoS'], yticklabels=['Benign','DDoS'])
ax.set_title(f'Phase 1 – Baseline IDS\nAccuracy: {ACC_BASE*100:.2f}%', fontsize=13)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

---
## Phase 2: Train 3 GAN Models

| # | GAN | Loss Function | Đặc điểm |
|---|---|---|---|
| 1 | **Vanilla GAN** | Binary Cross-Entropy | Baseline cơ bản |
| 2 | **WGAN-GP** | Wasserstein + Gradient Penalty | Ổn định hơn, tránh mode collapse |
| 3 | **LSGAN** | Least Squares (MSE) | Giảm vanishing gradient, ổn định hơn Vanilla |

> Sau khi train 3 GANs, so sánh chất lượng bằng KL Divergence và KS Test → chọn GAN tốt nhất.

In [ ]:
# ── Chuẩn bị dữ liệu cho GAN ─────────────────────────────────
gan_scaler = MinMaxScaler()
X_ddos_gan = gan_scaler.fit_transform(
    df_ddos_gan.values.astype('float32')
)

N_FEAT   = X_ddos_gan.shape[1]    # số features
NOISE    = 128                     # noise dimension
BATCH    = 256
EPOCHS   = 500

print(f'GAN training data : {X_ddos_gan.shape}')
print(f'Features          : {N_FEAT}')
print(f'Noise dim         : {NOISE}')
print(f'Epochs            : {EPOCHS}')

# ── Shared architectures (Generator & Discriminator/Critic) ─────
def make_generator(noise_dim, n_features):
    inp = tf.keras.Input(shape=(noise_dim,))
    x = layers.Dense(256)(inp)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512)(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256)(x)
    x = layers.LeakyReLU(0.2)(x)
    out = layers.Dense(n_features, activation='sigmoid')(x)
    return tf.keras.Model(inp, out, name='Generator')

def make_discriminator(n_features, sigmoid_out=True):
    inp = tf.keras.Input(shape=(n_features,))
    x = layers.Dense(256)(inp)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128)(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64)(x)
    x = layers.LeakyReLU(0.2)(x)
    activation = 'sigmoid' if sigmoid_out else None
    out = layers.Dense(1, activation=activation)(x)
    return tf.keras.Model(inp, out, name='Discriminator')

# Dataset helper
def make_dataset(X, batch_size):
    return (tf.data.Dataset
            .from_tensor_slices(X)
            .shuffle(10_000)
            .batch(batch_size, drop_remainder=True))

# Loss tracker helper
gan_results = {}   # lưu losses & samples của 3 GAN
print('✅ Shared utilities ready!')

### 2a. Vanilla GAN (Binary Cross-Entropy Loss)

In [ ]:
bce = tf.keras.losses.BinaryCrossentropy(from_logits=False)

# Build
vgan_gen = make_generator(NOISE, N_FEAT)
vgan_dis = make_discriminator(N_FEAT, sigmoid_out=True)
vgan_g_opt = Adam(2e-4, beta_1=0.5)
vgan_d_opt = Adam(2e-4, beta_1=0.5)

@tf.function
def vgan_step(real_batch):
    bsz = tf.shape(real_batch)[0]
    # Train Discriminator
    noise = tf.random.normal([bsz, NOISE])
    with tf.GradientTape() as dt:
        fake    = vgan_gen(noise, training=False)
        real_p  = vgan_dis(real_batch, training=True)
        fake_p  = vgan_dis(fake,       training=True)
        d_loss  = bce(tf.ones_like(real_p),  real_p) + \
                  bce(tf.zeros_like(fake_p), fake_p)
    d_grads = dt.gradient(d_loss, vgan_dis.trainable_variables)
    vgan_d_opt.apply_gradients(zip(d_grads, vgan_dis.trainable_variables))
    # Train Generator
    noise = tf.random.normal([bsz, NOISE])
    with tf.GradientTape() as gt:
        fake   = vgan_gen(noise, training=True)
        fake_p = vgan_dis(fake,  training=False)
        g_loss = bce(tf.ones_like(fake_p), fake_p)
    g_grads = gt.gradient(g_loss, vgan_gen.trainable_variables)
    vgan_g_opt.apply_gradients(zip(g_grads, vgan_gen.trainable_variables))
    return d_loss, g_loss

ds = make_dataset(X_ddos_gan, BATCH)
vg_losses, vd_losses = [], []
print(f'🚀 Vanilla GAN – {EPOCHS} epochs...')
for epoch in range(1, EPOCHS+1):
    gL, dL = [], []
    for batch in ds:
        d, g = vgan_step(batch)
        dL.append(float(d)); gL.append(float(g))
    vg_losses.append(np.mean(gL))
    vd_losses.append(np.mean(dL))
    if epoch % 100 == 0:
        print(f'  Epoch {epoch:4d} | D: {vd_losses[-1]:6.4f} | G: {vg_losses[-1]:6.4f}')

# Generate samples
vgan_samples = vgan_gen(tf.random.normal([10000, NOISE]), training=False).numpy()
gan_results['Vanilla GAN'] = {'g_losses': vg_losses, 'd_losses': vd_losses,
                              'samples': vgan_samples}
print('✅ Vanilla GAN done!')

# Loss plot
fig, (ax1,ax2) = plt.subplots(1,2, figsize=(12,4))
ax1.plot(vd_losses, color='#4c72b0'); ax1.set_title('Discriminator Loss'); ax1.set_xlabel('Epoch')
ax2.plot(vg_losses, color='#c44e52'); ax2.set_title('Generator Loss');     ax2.set_xlabel('Epoch')
plt.suptitle('Vanilla GAN Training', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 2b. WGAN-GP (Wasserstein + Gradient Penalty)

In [ ]:
LAMBDA_GP = 10
N_CRITIC  = 5

wgan_gen = make_generator(NOISE, N_FEAT)
wgan_crt = make_discriminator(N_FEAT, sigmoid_out=False)  # critic, no sigmoid
wgan_g_opt = Adam(1e-4, beta_1=0.0, beta_2=0.9)
wgan_c_opt = Adam(1e-4, beta_1=0.0, beta_2=0.9)

@tf.function
def gp_penalty(real, fake):
    alpha = tf.random.uniform([tf.shape(real)[0], 1], 0., 1.)
    interp = alpha * real + (1. - alpha) * fake
    with tf.GradientTape() as t:
        t.watch(interp)
        pred = wgan_crt(interp, training=True)
    grads = t.gradient(pred, interp)
    norm  = tf.sqrt(tf.reduce_sum(tf.square(grads), axis=1) + 1e-8)
    return tf.reduce_mean((norm - 1.) ** 2)

@tf.function
def wgan_step(real_batch):
    bsz = tf.shape(real_batch)[0]
    c_tot = tf.constant(0.0)                   # FIX: tf.constant thay vì Python float
    for _ in range(N_CRITIC):                  # FIX: range() thay vì tf.range() - tránh tf.while_loop
        noise = tf.random.normal([bsz, NOISE])
        with tf.GradientTape() as ct:
            fake   = wgan_gen(noise, training=False)
            r_out  = wgan_crt(real_batch, training=True)
            f_out  = wgan_crt(fake,       training=True)
            gp     = gp_penalty(real_batch, fake)
            c_loss = tf.reduce_mean(f_out) - tf.reduce_mean(r_out) + LAMBDA_GP * gp
        c_grads = ct.gradient(c_loss, wgan_crt.trainable_variables)
        wgan_c_opt.apply_gradients(zip(c_grads, wgan_crt.trainable_variables))
        c_tot = c_tot + c_loss                 # FIX: tường minh thay vì +=
    noise = tf.random.normal([bsz, NOISE])
    with tf.GradientTape() as gt:
        fake  = wgan_gen(noise, training=True)
        f_out = wgan_crt(fake,  training=False)  # FIX: training=False khi eval critic cho G
        g_loss = -tf.reduce_mean(f_out)
    g_grads = gt.gradient(g_loss, wgan_gen.trainable_variables)
    wgan_g_opt.apply_gradients(zip(g_grads, wgan_gen.trainable_variables))
    return c_tot / tf.cast(N_CRITIC, tf.float32), g_loss  # FIX: tf.cast để đảm bảo dtype

ds = make_dataset(X_ddos_gan, BATCH)
wg_losses, wc_losses = [], []
print(f'🚀 WGAN-GP – {EPOCHS} epochs...')
for epoch in range(1, EPOCHS+1):
    gL, cL = [], []
    for batch in ds:
        c, g = wgan_step(batch)
        cL.append(float(c)); gL.append(float(g))
    wg_losses.append(np.mean(gL))
    wc_losses.append(np.mean(cL))
    if epoch % 100 == 0:
        print(f'  Epoch {epoch:4d} | C: {wc_losses[-1]:7.4f} | G: {wg_losses[-1]:7.4f}')

wgan_samples = wgan_gen(tf.random.normal([10000, NOISE]), training=False).numpy()
gan_results['WGAN-GP'] = {'g_losses': wg_losses, 'd_losses': wc_losses,
                          'samples': wgan_samples}
print('✅ WGAN-GP done!')

fig, (ax1,ax2) = plt.subplots(1,2, figsize=(12,4))
ax1.plot(wc_losses, color='#4c72b0'); ax1.set_title('Critic Loss');    ax1.set_xlabel('Epoch')
ax2.plot(wg_losses, color='#c44e52'); ax2.set_title('Generator Loss'); ax2.set_xlabel('Epoch')
plt.suptitle('WGAN-GP Training', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

### 2c. LSGAN (Least Squares GAN)

> **Loss:** `D_loss = 0.5 * E[(D(real)-1)²] + 0.5 * E[D(fake)²]` | `G_loss = 0.5 * E[(D(fake)-1)²]`

> So với Vanilla GAN, LSGAN tránh được vanishing gradient và sinh mẫu có chất lượng tốt hơn.

In [ ]:
mse = tf.keras.losses.MeanSquaredError()

ls_gen = make_generator(NOISE, N_FEAT)
ls_dis = make_discriminator(N_FEAT, sigmoid_out=False)  # no sigmoid, output linear
ls_g_opt = Adam(2e-4, beta_1=0.5)
ls_d_opt = Adam(2e-4, beta_1=0.5)

@tf.function
def lsgan_step(real_batch):
    bsz = tf.shape(real_batch)[0]
    noise = tf.random.normal([bsz, NOISE])
    # Train Discriminator
    with tf.GradientTape() as dt:
        fake   = ls_gen(noise, training=False)
        r_out  = ls_dis(real_batch, training=True)
        f_out  = ls_dis(fake,       training=True)
        # LSGAN discriminator targets: real→1, fake→0
        d_loss = 0.5 * tf.reduce_mean(tf.square(r_out - 1.)) + \
                 0.5 * tf.reduce_mean(tf.square(f_out))
    d_grads = dt.gradient(d_loss, ls_dis.trainable_variables)
    ls_d_opt.apply_gradients(zip(d_grads, ls_dis.trainable_variables))
    # Train Generator
    noise = tf.random.normal([bsz, NOISE])
    with tf.GradientTape() as gt:
        fake  = ls_gen(noise,  training=True)
        f_out = ls_dis(fake,   training=False)
        # Generator target: fake → 1 (fool discriminator)
        g_loss = 0.5 * tf.reduce_mean(tf.square(f_out - 1.))
    g_grads = gt.gradient(g_loss, ls_gen.trainable_variables)
    ls_g_opt.apply_gradients(zip(g_grads, ls_gen.trainable_variables))
    return d_loss, g_loss

ds = make_dataset(X_ddos_gan, BATCH)
lg_losses, ld_losses = [], []
print(f'🚀 LSGAN – {EPOCHS} epochs...')
for epoch in range(1, EPOCHS+1):
    gL, dL = [], []
    for batch in ds:
        d, g = lsgan_step(batch)
        dL.append(float(d)); gL.append(float(g))
    lg_losses.append(np.mean(gL))
    ld_losses.append(np.mean(dL))
    if epoch % 100 == 0:
        print(f'  Epoch {epoch:4d} | D: {ld_losses[-1]:6.4f} | G: {lg_losses[-1]:6.4f}')

ls_samples = ls_gen(tf.random.normal([10000, NOISE]), training=False).numpy()
gan_results['LSGAN'] = {'g_losses': lg_losses, 'd_losses': ld_losses,
                        'samples': ls_samples}
print('✅ LSGAN done!')

fig, (ax1,ax2) = plt.subplots(1,2, figsize=(12,4))
ax1.plot(ld_losses, color='#4c72b0'); ax1.set_title('Discriminator Loss'); ax1.set_xlabel('Epoch')
ax2.plot(lg_losses, color='#c44e52'); ax2.set_title('Generator Loss');     ax2.set_xlabel('Epoch')
plt.suptitle('LSGAN Training', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## GAN Quality Comparison

So sánh 3 GAN bằng 2 chỉ số:
- **KL Divergence** (trung bình theo feature) → càng nhỏ càng tốt
- **KS Statistic** (Kolmogorov-Smirnov test) → càng nhỏ, phân phối GAN càng giống thật

In [ ]:
def kl_div(p, q, bins=50):
    """KL divergence between 1D distributions."""
    min_v = min(p.min(), q.min())
    max_v = max(p.max(), q.max()) + 1e-8
    p_hist, _ = np.histogram(p, bins=bins, range=(min_v, max_v), density=True)
    q_hist, _ = np.histogram(q, bins=bins, range=(min_v, max_v), density=True)
    p_hist = p_hist + 1e-8
    q_hist = q_hist + 1e-8
    return float(np.sum(rel_entr(p_hist, q_hist)))

real_orig = df_ddos_gan.values  # dữ liệu DDoS gốc (unscaled)

comparison = {}
for gan_name, result in gan_results.items():
    samples_orig = gan_scaler.inverse_transform(result['samples'])
    kl_scores, ks_scores = [], []
    for fi in range(N_FEAT):
        kl = kl_div(real_orig[:, fi], samples_orig[:, fi])
        ks = ks_2samp(real_orig[:, fi], samples_orig[:, fi]).statistic
        kl_scores.append(kl)
        ks_scores.append(ks)
    comparison[gan_name] = {
        'Mean KL Divergence': np.mean(kl_scores),
        'Mean KS Statistic' : np.mean(ks_scores),
    }

df_comp = pd.DataFrame(comparison).T
print('\n📊 GAN Quality Comparison:')
print(df_comp.round(4).to_string())

# Best GAN = lowest KL divergence
BEST_GAN = df_comp['Mean KL Divergence'].idxmin()
print(f'\n🏆 Best GAN: {BEST_GAN}')

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
metrics_plot = ['Mean KL Divergence', 'Mean KS Statistic']
gan_names = list(comparison.keys())
bar_colors = ['#4c72b0', '#dd8452', '#55a868']
for ai, metric in enumerate(metrics_plot):
    vals = [comparison[g][metric] for g in gan_names]
    bars = axes[ai].bar(gan_names, vals, color=bar_colors, edgecolor='k')
    for bar, v in zip(bars, vals):
        axes[ai].text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                      f'{v:.4f}', ha='center', fontweight='bold', fontsize=11)
    axes[ai].set_title(f'{metric}\n(lower = better)', fontsize=12)
    # Highlight best
    best_idx = np.argmin(vals)
    bars[best_idx].set_edgecolor('gold'); bars[best_idx].set_linewidth(3)
plt.suptitle(f'GAN Quality Comparison  |  🏆 Best: {BEST_GAN}',
             fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Phase 3: Generate Adversarial Samples with Best GAN

In [ ]:
N_SYNTH = 20_000
print(f'🔧 Generating {N_SYNTH:,} adversarial samples using {BEST_GAN}...')

# Chọn generator của best GAN
best_generators = {
    'Vanilla GAN': vgan_gen,
    'WGAN-GP'    : wgan_gen,
    'LSGAN'      : ls_gen,
}
best_gen = best_generators[BEST_GAN]

noise      = tf.random.normal([N_SYNTH, NOISE])
synth_sc   = best_gen(noise, training=False).numpy()
synth_orig = gan_scaler.inverse_transform(synth_sc)
df_synth   = pd.DataFrame(synth_orig, columns=SELECTED_FEATURES)

print(f'✅ Generated {len(df_synth):,} adversarial DDoS samples')

# PCA visualization: Real vs GAN DDoS
n_viz = min(2000, len(df_ddos_gan), len(df_synth))
real_v  = df_ddos_gan[SELECTED_FEATURES].sample(n_viz, random_state=42).values
synth_v = df_synth.sample(n_viz, random_state=42).values

pca = PCA(n_components=2, random_state=42)
proj = pca.fit_transform(np.vstack([real_v, synth_v]))
rp, sp = proj[:n_viz], proj[n_viz:]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(rp[:,0], rp[:,1], c='#c44e52', alpha=0.4, s=8, label='Real DDoS')
ax.scatter(sp[:,0], sp[:,1], c='#4c72b0', alpha=0.4, s=8, label=f'{BEST_GAN} (Adversarial)')
ax.set_title(f'PCA: Real vs {BEST_GAN} Adversarial DDoS', fontsize=13)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(markerscale=3, fontsize=11)
plt.tight_layout(); plt.show()

---
## Phase 4: Evaluate IDS on Adversarial GAN Traffic

> IDS (train trên real data) gặp traffic DDoS adversarial từ GAN.
> **Kỳ vọng:** Accuracy giảm xuống ~70% do GAN sinh mẫu có phân phối khác real DDoS.

In [ ]:
# ── Xây dựng adversarial test set ────────────────────────────────
X_test_benign = X_test_real[y_test_real == 0]
y_test_benign = y_test_real[y_test_real == 0]

n_adv = len(X_test_benign)
X_synth_sc = scaler.transform(
    df_synth[SELECTED_FEATURES].values[:n_adv]
)
y_synth_sc = np.ones(n_adv, dtype=int)

X_adv = np.vstack([X_test_benign, X_synth_sc])
y_adv = np.concatenate([y_test_benign, y_synth_sc])
idx = np.random.permutation(len(X_adv))
X_adv, y_adv = X_adv[idx], y_adv[idx]

print(f'Adversarial test set: {len(X_adv):,} samples')
print(f'   Benign   : {(y_adv==0).sum():,}')
print(f'   GAN-DDoS : {(y_adv==1).sum():,}')

# ── Đánh giá baseline IDS ─────────────────────────────────────────
y_pred_adv = ids_base.predict(X_adv)
y_prob_adv = ids_base.predict_proba(X_adv)[:, 1]

ACC_ADV = accuracy_score(y_adv, y_pred_adv)
F1_ADV  = f1_score(y_adv, y_pred_adv, average='macro')
AUC_ADV = roc_auc_score(y_adv, y_prob_adv)

print('\n' + '='*60)
print(f'PHASE 4: Baseline IDS vs {BEST_GAN} Adversarial Traffic')
print('='*60)
print(f'  Accuracy  : {ACC_ADV*100:.2f}%')
print(f'  F1 (macro): {F1_ADV:.4f}')
print(f'  AUC-ROC   : {AUC_ADV:.4f}')
print()
print(classification_report(y_adv, y_pred_adv, target_names=['Benign','GAN-DDoS']))

# Confusion matrix
fig, ax = plt.subplots(figsize=(6, 5))
cm2 = confusion_matrix(y_adv, y_pred_adv)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Reds', ax=ax,
            xticklabels=['Benign','GAN-DDoS'], yticklabels=['Benign','GAN-DDoS'])
ax.set_title(f'Phase 4 – Adversarial Test ({BEST_GAN})\nAccuracy: {ACC_ADV*100:.2f}%', fontsize=13)
ax.set_ylabel('Actual'); ax.set_xlabel('Predicted')
plt.tight_layout(); plt.show()

---
## Final Comparison: Before vs After GAN Attack

In [ ]:
# ── Confusion matrices side-by-side ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm1 = confusion_matrix(y_test_real, y_pred_base)

sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Benign','DDoS'], yticklabels=['Benign','DDoS'])
axes[0].set_title(f'Phase 1 – Real Test\nAccuracy: {ACC_BASE*100:.2f}%', fontsize=12)
axes[0].set_ylabel('Actual'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm2, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Benign','GAN-DDoS'], yticklabels=['Benign','GAN-DDoS'])
axes[1].set_title(f'Phase 4 – {BEST_GAN} Adversarial\nAccuracy: {ACC_ADV*100:.2f}%', fontsize=12)
axes[1].set_ylabel('Actual'); axes[1].set_xlabel('Predicted')

plt.suptitle('Confusion Matrix: Before vs After GAN Attack', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ── Bar chart ─────────────────────────────────────────────────────
metrics_names = ['Accuracy', 'F1 (macro)', 'AUC-ROC']
phase1_vals   = [ACC_BASE, F1_BASE, AUC_BASE]
phase4_vals   = [ACC_ADV,  F1_ADV,  AUC_ADV]

x = np.arange(len(metrics_names)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x-w/2, phase1_vals, w, label='Phase 1 (Real Data)',
            color='#4c72b0', edgecolor='k')
b2 = ax.bar(x+w/2, phase4_vals, w, label=f'Phase 4 ({BEST_GAN} Adversarial)',
            color='#c44e52', edgecolor='k')
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.006,
            f'{bar.get_height():.4f}', ha='center', fontsize=10, fontweight='bold')
ax.axhline(0.90, color='green',  ls='--', alpha=0.7, label='90% line')
ax.axhline(0.70, color='orange', ls='--', alpha=0.7, label='70% line')
ax.set_ylim(0, 1.12); ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('IDS Performance: Before vs After GAN Adversarial Attack',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('comparison_before_after_gan.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Final summary table ───────────────────────────────────────────
drop_acc = (ACC_BASE - ACC_ADV) * 100
drop_f1  = (F1_BASE  - F1_ADV)  * 100

summary = pd.DataFrame({
    'Phase'       : ['Phase 1 – Real Data', f'Phase 4 – {BEST_GAN} Adversarial'],
    'Accuracy (%)': [round(ACC_BASE*100,2),  round(ACC_ADV*100,2)],
    'F1 macro'    : [round(F1_BASE,4),        round(F1_ADV,4)],
    'AUC-ROC'     : [round(AUC_BASE,4),       round(AUC_ADV,4)],
})
print('\n' + '='*65)
print('FINAL SUMMARY')
print('='*65)
print(summary.to_string(index=False))
print(f'\n📉 Accuracy drop : {ACC_BASE*100:.2f}% → {ACC_ADV*100:.2f}% (↓{drop_acc:.2f}%)')
print(f'   F1 drop       : {F1_BASE:.4f} → {F1_ADV:.4f} (↓{drop_f1:.2f}%)')
print(f'\n🏆 Best GAN      : {BEST_GAN}')
print(f'\n🔑 Kết luận:')
print(f'   • {BEST_GAN} sinh được mẫu DDoS adversarial có chất lượng cao nhất')
print(f'     (KL Divergence thấp nhất, phân phối gần thật nhất)')
print(f'   • IDS baseline bị sụt giảm accuracy {drop_acc:.2f}% khi gặp GAN adversarial')
print(f'   • Điều này chứng minh GAN có khả năng sinh traffic evasive,')
print(f'     vượt qua IDS truyền thống được huấn luyện trên dữ liệu thật.')